# Twitch Loyalty Story Notebook
Exploring how Twitch streamers turn diverse audiences into loyal superfans.
        


## Outline
1. Load cleaned interaction data.
2. Aggregate channel-level loyalty metrics.
3. Quantify audience-cluster diversity and fan concentration.
4. Visualize the hero insight plus supporting evidence.
        


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 150, 'axes.titlesize': 14, 'axes.labelsize': 12, 'font.size': 11})
        


In [ ]:
DATA_PATH = Path('data/cleaned.csv')
raw = pd.read_csv(DATA_PATH)
raw.head()
        


In [ ]:
def build_channel_features(df: pd.DataFrame) -> pd.DataFrame:
    channel = df.groupby('channel_login').agg(
        total_interactions=('viewer_id','size'),
        unique_viewers=('viewer_id','nunique'),
        session_count=('session_id','nunique'),
        language_diversity=('language_cluster_id','nunique')
    ).reset_index()
    channel['loyalty_ratio'] = channel['total_interactions'] / channel['unique_viewers'].clip(lower=1)
    channel['tier'] = pd.cut(
        channel['unique_viewers'],
        bins=[0, 50, 500, channel['unique_viewers'].max()+1],
        labels=['Emerging (<50)', 'Mid-tier (50-499)', 'Mega (500+)'],
        include_lowest=True,
        right=False
    )
    lang = df.groupby(['channel_login','language_cluster_id']).size().reset_index(name='interaction_count')
    top_lang = lang.sort_values('interaction_count', ascending=False).drop_duplicates('channel_login')
    top_lang = top_lang.rename(columns={'language_cluster_id':'top_language_cluster','interaction_count':'top_cluster_interactions'})
    channel = channel.merge(top_lang, on='channel_login', how='left')
    channel['community_concentration'] = channel['top_cluster_interactions'] / channel['total_interactions']
    return channel

def compute_top10_share(df: pd.DataFrame) -> pd.DataFrame:
    viewer_channel = df.groupby(['channel_login','viewer_id']).size().reset_index(name='viewer_channel_interactions')
    totals = viewer_channel.groupby('channel_login')['viewer_channel_interactions'].sum().reset_index(name='total_interactions')
    viewer_channel = viewer_channel.sort_values(['channel_login','viewer_channel_interactions'], ascending=[True, False])
    viewer_channel['rank'] = viewer_channel.groupby('channel_login')['viewer_channel_interactions'].rank(method='first', ascending=False)
    top10 = viewer_channel[viewer_channel['rank'] <= 10]
    top10_sum = top10.groupby('channel_login')['viewer_channel_interactions'].sum().reset_index(name='top10_interactions')
    share = totals.merge(top10_sum, on='channel_login', how='left').fillna({'top10_interactions':0})
    share['top10_share'] = share['top10_interactions'] / share['total_interactions']
    return share[['channel_login','top10_share']]

channel_features = build_channel_features(raw)
channel_features = channel_features.merge(compute_top10_share(raw), on='channel_login', how='left')
channel_features.head()
        


In [ ]:
bands = pd.cut(
    channel_features['language_diversity'],
    bins=[0,3,10,50,100,channel_features['language_diversity'].max()+1],
    labels=['1-3','4-10','11-50','51-100','101+'],
    include_lowest=True
)
loyal_summary = channel_features.groupby(bands).agg(
    median_loyalty=('loyalty_ratio','median'),
    channels=('channel_login','count')
).reset_index()
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(loyal_summary['language_diversity'], loyal_summary['median_loyalty'], marker='o')
ax.set_ylabel('Median interactions per viewer')
ax.set_title('Global Variety Doubles Viewer Loyalty', loc='left')
for _, row in loyal_summary.iterrows():
    ax.annotate('{:.2f}× (n={:,})'.format(row['median_loyalty'], int(row['channels'])), (row['language_diversity'], row['median_loyalty']), textcoords='offset points', xytext=(0,10), ha='center')
plt.show()
        


In [ ]:
fan_summary = channel_features.groupby('tier').agg(
    median_top10_share=('top10_share','median')
).reset_index()
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(fan_summary['tier'], fan_summary['median_top10_share']*100)
ax.set_ylabel('Median % of interactions from top 10 viewers')
ax.set_title('Mid-tier Streamers Depend on Superfans', loc='left')
for idx, row in fan_summary.iterrows():
    ax.text(idx, row['median_top10_share']*100 + 1, '{:.1f}%'.format(row['median_top10_share']*100), ha='center')
plt.show()
        


In [ ]:
viewer_channel = raw.groupby(['viewer_id','channel_login']).size().reset_index(name='viewer_channel_interactions')
viewer_stats = viewer_channel.groupby('viewer_id')['channel_login'].nunique().reset_index(name='channels_followed')
median_channels = viewer_stats['channels_followed'].median()
p90 = viewer_stats['channels_followed'].quantile(0.9)
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(viewer_stats['channels_followed'], bins=30, color='#20A39E', alpha=0.85)
ax.axvline(median_channels, color='#7C5DFA', linestyle='--', label='Median {:.0f}'.format(median_channels))
ax.axvline(p90, color='#FFB347', linestyle=':', label='90th pct {:.0f}'.format(p90))
ax.set_xlabel('Distinct channels per viewer')
ax.set_ylabel('Viewer count')
ax.legend()
ax.set_title('Viewers Graze Across Hundreds of Channels', loc='left')
plt.show()
        
